In [6]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

df = pd.read_csv("../data/processed/marketing_clean.csv", parse_dates=["date", "month"])
RAW_COLS = ["impressions", "clicks", "ad_spend", "conversions", "revenue"]
FUNNEL = ["impressions", "clicks", "conversions"]


def safe_div(a, b):
    return a.div(b.where(b != 0))


def add_metrics(g):
    g = g.copy()
    g["CTR"] = safe_div(g["clicks"], g["impressions"])
    g["CPC"] = safe_div(g["ad_spend"], g["clicks"])
    g["conv_rate"] = safe_div(g["conversions"], g["clicks"])
    g["CAC"] = safe_div(g["ad_spend"], g["conversions"])
    g["ROAS"] = safe_div(g["revenue"], g["ad_spend"])
    return g

In [2]:
def build_funnel(keys):
    g = df[FUNNEL].groupby(keys).sum()
    name = keys.name
    rows = []
    for key, r in g.iterrows():
        prev = None
        for order, stage in enumerate(FUNNEL, start=1):
            val = r[stage]
            rows.append({
                name: key,
                "stage_order": order,
                "stage": stage,
                "value": val,
                "pct_of_impressions": val / r["impressions"],
                "step_rate": val / prev if prev else np.nan,
                "step_loss": 1 - val / prev if prev else np.nan,
            })
            prev = val
    return pd.DataFrame(rows)


by_platform_funnel = build_funnel(df["platform"])
overall_funnel = build_funnel(pd.Series("All", index=df.index, name="platform"))
funnel = pd.concat([overall_funnel, by_platform_funnel], ignore_index=True)
funnel

,platform,stage_order,stage,value,pct_of_impressions,step_rate,step_loss
0,All,1,impressions,185254234,1.0000,NaN,NaN
1,All,2,clicks,7132816,0.0385,0.0385,0.9615
2,All,3,conversions,326812,0.0018,0.0458,0.9542
3,Google Ads,1,impressions,73801150,1.0000,NaN,NaN
4,Google Ads,2,clicks,2936149,0.0398,0.0398,0.9602
5,Google Ads,3,conversions,131098,0.0018,0.0446,0.9554
6,Meta Ads,1,impressions,64403494,1.0000,NaN,NaN
7,Meta Ads,2,clicks,1596736,0.0248,0.0248,0.9752
8,Meta Ads,3,conversions,73262,0.0011,0.0459,0.9541
9,TikTok Ads,1,impressions,47049590,1.0000,NaN,NaN


In [3]:
wide = funnel.pivot(index="platform", columns="stage", values="step_loss")[["clicks", "conversions"]]
wide.columns = ["loss_impression_to_click", "loss_click_to_conversion"]
wide["end_to_end_rate"] = funnel[funnel["stage"] == "conversions"].set_index("platform")["pct_of_impressions"]
wide

,loss_impression_to_click,loss_click_to_conversion,end_to_end_rate
platform,,,
All,0.9615,0.9542,0.0018
Google Ads,0.9602,0.9554,0.0018
Meta Ads,0.9752,0.9541,0.0011
TikTok Ads,0.9447,0.9529,0.0026


In [4]:
base = add_metrics(df.groupby("platform")[RAW_COLS].sum())[["ad_spend", "revenue", "ROAS"]]
total_revenue = base["revenue"].sum()
total_spend = base["ad_spend"].sum()
SOURCE = "Google Ads"

destinations = {
    "TikTok only": {"TikTok Ads": 1.0},
    "TikTok + Meta 50/50": {"TikTok Ads": 0.5, "Meta Ads": 0.5},
}


def run_scenario(shift, weights, haircut):
    moved = base.loc[SOURCE, "ad_spend"] * shift
    lost = moved * base.loc[SOURCE, "ROAS"]
    gained = sum(moved * w * base.loc[p, "ROAS"] * (1 - haircut) for p, w in weights.items())
    new_revenue = total_revenue - lost + gained
    return {
        "moved_spend": moved,
        "new_revenue": new_revenue,
        "delta_revenue": new_revenue - total_revenue,
        "delta_pct": new_revenue / total_revenue - 1,
        "new_ROAS": new_revenue / total_spend,
    }


rows = []
for dest_name, weights in destinations.items():
    for shift in [0.10, 0.20, 0.30]:
        for haircut in [0.0, 0.25, 0.50]:
            res = run_scenario(shift, weights, haircut)
            res.update({
                "destination": dest_name,
                "shift_from_google": shift,
                "marginal_haircut": haircut,
                "max_dest_spend_increase": max(
                    res["moved_spend"] * w / base.loc[p, "ad_spend"] for p, w in weights.items()
                ),
            })
            rows.append(res)

scenarios = pd.DataFrame(rows)[[
    "destination", "shift_from_google", "marginal_haircut", "moved_spend",
    "max_dest_spend_increase", "new_revenue", "delta_revenue", "delta_pct", "new_ROAS",
]]
scenarios

,destination,shift_from_google,marginal_haircut,moved_spend,max_dest_spend_increase,new_revenue,delta_revenue,delta_pct,new_ROAS
0,TikTok only,0.1000,0.0000,"634,926.8910",0.2393,"56,819,173.8652","2,635,843.0552",0.0486,5.1148
1,TikTok only,0.1000,0.2500,"634,926.8910",0.2393,"55,609,369.4777","1,426,038.6677",0.0263,5.0059
2,TikTok only,0.1000,0.5000,"634,926.8910",0.2393,"54,399,565.0901","216,234.2801",0.0040,4.8970
3,TikTok only,0.2000,0.0000,"1,269,853.7820",0.4786,"59,455,016.9204","5,271,686.1104",0.0973,5.3521
4,TikTok only,0.2000,0.2500,"1,269,853.7820",0.4786,"57,035,408.1453","2,852,077.3353",0.0526,5.1343
5,TikTok only,0.2000,0.5000,"1,269,853.7820",0.4786,"54,615,799.3702","432,468.5602",0.0080,4.9165
6,TikTok only,0.3000,0.0000,"1,904,780.6730",0.7179,"62,090,859.9756","7,907,529.1656",0.1459,5.5894
7,TikTok only,0.3000,0.2500,"1,904,780.6730",0.7179,"58,461,446.8130","4,278,116.0030",0.0790,5.2626
8,TikTok only,0.3000,0.5000,"1,904,780.6730",0.7179,"54,832,033.6503","648,702.8403",0.0120,4.9359
9,TikTok + Meta 50/50,0.1000,0.0000,"634,926.8910",0.1507,"56,197,272.9358","2,013,942.1258",0.0372,5.0588


In [7]:
for p in ["TikTok Ads", "Meta Ads"]:
    be = 1 - base.loc[SOURCE, "ROAS"] / base.loc[p, "ROAS"]
    print(f"{p}: break-even haircut = {be:.0%}")

TikTok Ads: break-even haircut = 54%
Meta Ads: break-even haircut = 39%


In [8]:
OUT_DIR = "../data/processed/"
funnel.to_csv(OUT_DIR + "funnel_platform.csv", index=False)
scenarios.to_csv(OUT_DIR + "budget_scenarios.csv", index=False)
print(funnel.shape, scenarios.shape)

(12, 7) (18, 9)


- Funnel: post-click loss is ~95.3-95.5% on every platform; platforms differ only in CTR (TikTok 5.53%, Google 3.98%, Meta 2.48%), not in conv. rate (4.46-4.71%)
- ROAS gap comes from CPC (TikTok 1.02, Meta 1.32, Google 2.16): ROAS TikTok 7.62 > Meta 5.66 > Google 3.47; CAC 21.67 / 28.75 / 48.43
- Ratios are computed from sums: aggregate ROAS 4.88 vs mean of row ROAS 6.45 (median 4.30)
- Shifting 20% of Google budget to TikTok + Meta: +7.4% revenue (no diminishing returns), +3.5% at 25% haircut, -0.4% at 50%
- Break-even haircut: TikTok 54.5%, Meta 38.7%
- Recommendation: shift in stages (10% first, then 20%); don't shut Google down (41% of revenue)
- ROAS is not profit (break-even ROAS = 1 / margin)